# Lab 5.2 &mdash; LangGraph: the AskOps triage graph

**About 25 minutes** &middot; Day 2 &middot; Module 5 &mdash; LangChain &amp; LangGraph

In this lab you draw AskOps as a graph. The model decides whether a question is an outage or a how-to-fix question. A search that finds nothing goes round a **cycle**: the model rewrites the search words and tries again, at most three times.

Run the cells in order, with **Shift + Enter**. Under each cell, **You should see** says what to
expect. The model is real, so its words change from run to run. The shape of the result does not.

**The result:** three questions take three different paths through your graph, and you watch each step as it happens.

```
classify --runbook--> search --<found_enough?>--found--> answer
   |                    ^          |
   |                    +-rewrite--+  (nothing found, fewer than 3 tries)
   +--incident--> read_incidents -----------------> answer
```

`<found_enough?>` is not a node. It is a router: a function that picks the next node.

## Step 1 &mdash; The Lab 5.1 agent is already a graph

`create_agent` built a small LangGraph graph for you. Print its nodes and edges. In this lab you
draw a graph like it yourself, and choose every node and edge.

In [ ]:
from langchain.agents import create_agent
from askops import get_llm, langchain_tools

llm = get_llm()
agent = create_agent(llm, langchain_tools()[:3])
print("nodes:", list(agent.get_graph().nodes))
for e in agent.get_graph().edges:
    print(f"  {e.source} -> {e.target}")

**You should see:** two real nodes, `model` and `tools`, plus `__start__` and `__end__`. The model
decides whether to go to `tools` or to the end, and `tools` always goes back to `model`. That is
the Module 4 loop: reason, act, observe, as a graph.

## Step 2 &mdash; See why reducers matter

A graph has one **state**: a dictionary that every node reads. A node returns only the keys it
changed, and LangGraph joins them into the state. The **reducer** decides how. With no reducer, a new
value **replaces** the old one. With `Annotated[list, add]`, a new list is **added** to the old one.

This tiny graph has two nodes. Each one writes a note. Watch what survives.

In [ ]:
import json
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from askops import get_llm, search_runbooks, list_incidents

llm = get_llm()

def two_notes(state_type):
    g = StateGraph(state_type)
    g.add_node("first", lambda s: {"notes": ["from first"]})
    g.add_node("second", lambda s: {"notes": ["from second"]})
    g.add_edge(START, "first"); g.add_edge("first", "second"); g.add_edge("second", END)
    return g.compile().invoke({"notes": []})["notes"]

class Replaced(TypedDict):
    notes: list
class Added(TypedDict):
    notes: Annotated[list, add]

print("no reducer  :", two_notes(Replaced))
print("with reducer:", two_notes(Added))

**You should see:** with no reducer, only `from second` is left. The first note was lost, with no
error. With the reducer, both notes are kept. Remember this when two nodes write the same key.

## Step 3 &mdash; The state and the nodes

Each node is a plain Python function. It gets the state and returns **only the keys it changed**.
Two nodes ask the model: `classify` picks the path, and `rewrite` suggests new search words. It is
shown the searches that already failed, so each try uses different words.

In [ ]:
class AskState(TypedDict):
    question: str
    kind: str                            # "runbook" or "incident"
    query: str                           # the words search uses
    findings: Annotated[list, add]       # every node adds to this list
    tries: int
    answer: str

def classify(s):
    word = llm.invoke("Classify the on-call question. Reply with one word: incident or runbook.\n"
                      "Say incident ONLY if it says a whole service is down, has an outage, or cannot "
                      "be reached at all.\nSay runbook for everything else: errors, failures, slowness, "
                      "or how to fix something.\n\n" + s["question"]).content.lower()
    return {"kind": "incident" if "incident" in word else "runbook", "query": s["question"]}

def search(s):
    hits = json.loads(search_runbooks(s["query"]))
    found = ", ".join(f"{h['id']} ({h['title']})" for h in hits) or "nothing"
    return {"findings": [f"search '{s['query']}': {found}"], "tries": s["tries"] + 1}

def rewrite(s):
    tried = "\n".join(s["findings"])             # the searches that found nothing
    words = llm.invoke("Give 3 short search words for an ops runbook about this problem, such as "
                       "'disk full cleanup'. Use words that differ from the searches below, which "
                       "found nothing. Reply with 3 words on one line.\n\nProblem: " + s["question"]
                       + "\nSearches tried:\n" + tried).content
    return {"query": " ".join(words.replace('"', "").split()[:3])}   # one line, 3 words

def read_incidents(s):
    ids = [f"{i['id']} ({i['service']}: {i['title']})" for i in json.loads(list_incidents())]
    return {"findings": ["open incidents: " + ", ".join(ids)]}

def answer(s):
    text = llm.invoke("Answer the on-call engineer in at most 3 lines, only from these findings. "
                      "Cite only a runbook or incident whose title describes the same problem. The "
                      "same service is not enough. If nothing fits, say so and suggest a hand-over "
                      "to a person.\n\nQuestion: " + s["question"]
                      + "\nFindings:\n" + "\n".join(s["findings"])).content
    return {"answer": text}

# A node is just a function, so you can test it on its own:
print(search({"query": "502 after deploy", "tries": 0}))

**You should see:** `{'findings': ["search '502 after deploy': RB-101 (Payments API returns 502 after deploy)"], 'tries': 1}`. The node
returned two keys, not the whole state. The finding carries the runbook title, so `answer` can
judge whether it fits.

## Step 4 &mdash; Edges, routers and a cycle

A **plain edge** always goes to the same node. A **conditional edge** calls a **router**: a function
that reads the state and returns the name of the next node. The edge from `rewrite` back to `search`
makes the **cycle**. The router stops it after three tries, and `recursion_limit` is the backup.

In [ ]:
def route(s):
    return "search" if s["kind"] == "runbook" else "read_incidents"

def found_enough(s):
    found = "RB-" in s["findings"][-1]
    return "answer" if found or s["tries"] >= 3 else "rewrite"

g = StateGraph(AskState)
for name, fn in [("classify", classify), ("search", search), ("rewrite", rewrite),
                 ("read_incidents", read_incidents), ("answer", answer)]:
    g.add_node(name, fn)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route, ["search", "read_incidents"])
g.add_conditional_edges("search", found_enough, ["answer", "rewrite"])
g.add_edge("rewrite", "search")
g.add_edge("read_incidents", "answer")
g.add_edge("answer", END)
graph = g.compile()
print("nodes:", [n for n in graph.get_graph().nodes if not n.startswith("__")])

**You should see:** the five node names. The graph is built and checked, and nothing has run yet.

## The result &mdash; three questions, three paths

`stream_mode="updates"` prints each node as it finishes, with only the keys it changed. Watch the path
each question takes.

In [ ]:
QUESTIONS = [
    "Payments returns 502 after deploy. What do I do?",           # found on the first search
    "Customers get errors when paying since this afternoon",       # needs better search words
    "The auth service is down for everyone right now",             # an outage
]

for q in QUESTIONS:
    print("Q:", q)
    start = {"question": q, "findings": [], "tries": 0}
    for update in graph.stream(start, {"recursion_limit": 15}, stream_mode="updates"):
        for node, changes in update.items():
            if node == "answer":
                print(f"  answer          -> {changes['answer']}")
            else:
                print(f"  {node:<15} -> {changes}")
    print("-" * 70)

**You should see:**

- The **502 question** goes `classify`, `search`, `answer`, and the answer cites **RB-101**.
- The **paying question** finds nothing with its own words. It goes through `rewrite` and searches
  again with the model's words. That search may find three runbooks, such as RB-101, RB-102 and
  RB-301. The answer drops RB-301: its title shows it is about a nightly batch job. That is why
  `search` keeps the titles in the findings. If nothing is found after three tries, the answer hands
  over to a person.
- The **outage** goes `classify`, `read_incidents`, `answer`. The only open auth incident is
  INC-9002, and it is about slow logins, not an outage. The answer should say so and hand over.

The model may still name INC-9002 first, although the prompt says the same service is not enough.
Would that answer be safe for an engineer on call at 3 a.m.? Write your view in your notes.

Each line shows only the keys that node changed. That is the rule from Step 3, now visible.